## Prediccion de la produccion industrial por sectores - Luna Luciano

## 1. Vista previa del dataset original (crudo)

A continuación se muestra una porción del archivo original de producción industrial. Este archivo en Excel contiene una hoja por sector, con formatos distintos, encabezados desplazados y estructuras inconsistentes. Por este motivo fue necesario aplicar una limpieza personalizada por hoja.


In [ ]:
import pandas as pd

# Visualización parcial de una hoja cruda del Excel original
archivo_crudo = pd.read_excel('../data/raw/14_3_03_Produccion_Industrial-1-1.xlsx', sheet_name='CONFECCIONISTA')
archivo_crudo.head(10)


## 2. Transformaciones aplicadas al dataset de producción

Sobre este archivo se aplicaron las siguientes tareas de limpieza y normalización:

- Recorte y renombrado de columnas
- Conversión a formato largo con nombre de sector
- Normalización de nombres de meses y sectores
- Conversión de tipos de datos (`anio`, `Producción`)
- Imputación de valores faltantes y reemplazo de símbolos inválidos
- Orden cronológico por sector, año y mes


## 3. Dataset de producción limpio y estructurado

Este es el resultado final tras aplicar todas las transformaciones. El dataset se encuentra normalizado por `anio`, `mes`, `sector` y contiene la columna `Produccion` lista para ser utilizada en el análisis y modelado.


In [ ]:
df_produccion_total = pd.read_csv('../data/processed/produccion_total_por_sector_v2.csv')
df_produccion_total

## 4. Vista previa de los otros datasets procesados

Los datasets de empleados y establecimientos fueron tratados con un proceso de limpieza similar:

- Selección de columnas útiles
- Conversión a formato largo
- Normalización de nombres de sectores y meses
- Conversión de tipos de datos
- Imputación manual en casos específicos del sector Pesqueras

A continuación, se muestran las versiones ya limpias, listas para ser unificadas.


### Comparativa: antes y después de la limpieza

A continuación se muestra un ejemplo visual de cómo eran originalmente los archivos de empleados y establecimientos, y cómo quedaron luego del proceso de limpieza y transformación.


In [ ]:
# Empleados - Vista previa cruda
empleados_crudo = pd.read_excel('../data/raw/14_3_01_Personal_industria_rama-1.xlsx')
empleados_crudo.head(10)


In [ ]:
# Empleados – luego de la limpieza
df_empleados_limpio = pd.read_csv('../data/processed/empleados_total_por_sector_v2.csv')
df_empleados_limpio.head()


In [ ]:
# Establecimientos – vista cruda
estab_crudo = pd.read_excel('../data/raw/14_3_02_Establecimientos_industriales_rama-1.xlsx')
estab_crudo.head(10)


In [ ]:
# Establecimientos – luego de la limpieza
df_estab_limpio = pd.read_csv('../data/processed/establecimientos_total_por_sector_v2.csv')
df_estab_limpio.head()


## 5. Unificación de los datasets y separación para modelado

Una vez que los tres datasets (`producción`, `empleados`, `establecimientos`) fueron limpiados y normalizados, realicé la unificación completa utilizando las claves `anio`, `mes` y `sector`.

Para no perder información histórica, utilicé una unión externa (`outer join`), lo que me permitió conservar registros aunque no tuvieran valor en la variable objetivo (`Producción`).

Esto me permitió dividir los datos en:
- **`df_final`**: con registros completos (listo para entrenar y evaluar modelos)
- **`df_test`**: con producción faltante (usado luego para predicción real)


In [ ]:
# Carga de datasets procesados
df_unificado = pd.read_csv('../data/processed/dataset_unificado_industria_v2.csv')
df_final = pd.read_csv('../data/processed/dataset_final.csv')
df_test = pd.read_csv('../data/processed/dataset_test_real.csv')

# Vista previa
print("Dataset unificado:")
display(df_unificado.head())

print("\nDataset final para modelado:")
display(df_final.head())

print("\nDataset reservado para predicción:")
display(df_test.head())


## 6. Análisis exploratorio y división por unidad de medida

Al analizar el dataset `df_final`, se identificó que los sectores industriales reportan la producción en diferentes unidades.  
Para evitar mezclar magnitudes incompatibles, se dividió el dataset en:

- `df_unidades`: producción en unidades físicas (Confeccionista, Electrónica, Otros)
- `df_kilos`: producción en kilogramos (Textil, Plástica, Pesquera)

---

### Análisis exploratorio sobre `df_unidades`

Se analizó la producción mensual por sector y su relación con las variables independientes (`empleados`, `establecimientos`).

- **Electrónica** mostró una fuerte correlación positiva con la producción y mantiene valores altos y estables hasta 2016, con una leve disminución posterior.
- **Confeccionista** presentó una relación más moderada pero relativamente estable a lo largo del tiempo.
- **Otros** exhibió un comportamiento errático, con picos extremos al inicio y una caída abrupta a partir de 2016, lo que genera ruido en el análisis.

Por este motivo, se excluyó el sector "Otros" para entrenar modelos más coherentes.


In [ ]:
# División por unidad de medida
sectores_unidades = ['Confeccionista', 'Electronica', 'Otros']
sectores_kilos = ['Textil', 'Plastica', 'Pesquera']

df_unidades = df_final[df_final['sector'].isin(sectores_unidades)].reset_index(drop=True)
df_kilos = df_final[df_final['sector'].isin(sectores_kilos)].reset_index(drop=True)

# Refinamiento: elimino sector "Otros"
sectores_conf_elec = ['Confeccionista', 'Electronica']
df_unidades_conf_elec = df_unidades[df_unidades['sector'].isin(sectores_conf_elec)].reset_index(drop=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Orden de los meses para asegurar consistencia temporal
orden_meses = [
    'enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
    'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre'
]

# Aseguro el orden y genero la columna 'fecha'
df_unidades['mes'] = pd.Categorical(df_unidades['mes'], categories=orden_meses, ordered=True)
df_unidades['fecha'] = pd.to_datetime(df_unidades['anio'].astype(str) + '-' + (df_unidades['mes'].cat.codes + 1).astype(str), errors='coerce')


plt.figure(figsize=(10, 6))
sns.lineplot(data=df_unidades, x='fecha', y='Produccion', hue='sector')
plt.title("Evolución mensual de la Producción (sectores en unidades)")
plt.xlabel("Fecha")
plt.ylabel("Producción (en unidades)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación empleados vs Producción por sector (resumen final)
df_unidades.groupby('sector')[['empleados', 'Produccion']].corr().iloc[::2, -1]


In [ ]:
# Mapa de correlación para Confeccionista y Electrónica
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df_unidades[['empleados', 'establecimientos', 'Produccion']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación - Sectores en unidades")
plt.tight_layout()
plt.show()


#### Mapa de correlación para Confeccionista y Electrónica

Se genera un subconjunto con los sectores Confeccionista y Electrónica para evitar que Otros distorsione las relaciones. A continuación se visualiza la matriz de correlación entre variables numéricas para estos sectores.

In [ ]:
# Mapa de correlación para Confeccionista y Electrónica
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df_unidades_conf_elec[['empleados', 'establecimientos', 'Produccion']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación - Sectores en unidades")
plt.tight_layout()
plt.show()


### Análisis exploratorio sobre `df_kilos`

Se analizó la producción mensual por sector y su relación con las variables independientes (*empleados*, *establecimientos*).

- **Textil** mostró una correlación positiva clara con la producción y una tendencia decreciente desde 2018.
- **Pesquera** tuvo una relación moderada y más estable con la producción.
- **Plástica** presentó una correlación negativa y gran dispersión, lo que sugiere un comportamiento productivo atípico.

Por este motivo, se excluyó el sector "Plástica" para entrenar modelos más coherentes.



In [ ]:
# Orden de los meses para asegurar consistencia temporal
orden_meses = [
    'enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
    'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre'
]

# Aseguro el orden y genero la columna 'fecha'
df_kilos['mes'] = pd.Categorical(df_kilos['mes'], categories=orden_meses, ordered=True)
df_kilos['fecha'] = pd.to_datetime(df_kilos['anio'].astype(str) + '-' + (df_kilos['mes'].cat.codes + 1).astype(str), errors='coerce')


plt.figure(figsize=(10, 6))
sns.lineplot(data=df_kilos, x='fecha', y='Produccion', hue='sector')
plt.title("Evolución mensual de la Producción (sectores en kg)")
plt.xlabel("Fecha")
plt.ylabel("Producción (en kg)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación empleados vs Producción por sector (resumen final)
df_kilos.groupby('sector')[['empleados', 'Produccion']].corr().iloc[::2, -1]


In [ ]:
corr_matrix = df_kilos[['empleados', 'establecimientos', 'Produccion']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de correlación - Sectores en Kg")
plt.tight_layout()
plt.show()

#### Mapa de correlación para Textil y Pesquera

Se genera un subconjunto con los sectores Textil y Pesquera para evitar que Plástica distorsione las relaciones. A continuación se visualiza la matriz de correlación entre variables numéricas para estos sectores.


In [ ]:
# Subconjunto sin Plástica
sectores_kilos_filtrados = ['Textil', 'Pesquera']
df_kilos_text_pesq = df_kilos[df_kilos['sector'].isin(sectores_kilos_filtrados)].copy()

# Mapa de correlación
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df_kilos_text_pesq[['empleados', 'establecimientos', 'Produccion']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de correlación - Sectores en Kg")
plt.tight_layout()
plt.show()


## 7. Modelado de regresión supervisada

En esta sección se busca predecir la **producción mensual por sector industrial** utilizando modelos de regresión supervisada, entrenados con variables explicativas de tipo estructural y temporal.

Para lograrlo, se trabajó con un enfoque progresivo dividido en tres fases principales:

- **Fase 1**: Modelos lineales clásicos y regularizados (Ridge, Lasso, ElasticNet).
- **Fase 2**: Modelos no lineales (árboles de decisión, bosques aleatorios, vecinos más cercanos).
- **Fase 3**: Validación cruzada para evaluar estabilidad y robustez.

Se entrenaron modelos **por sector** y modelos **generales** que combinan sectores con la misma unidad de medida.  
> Para este notebook final, y por cuestiones de tiempo, se mostrarán únicamente los **modelos generales**, que demostraron un excelente desempeño y permiten explicar resultados de manera compacta y representativa.

Los modelos individuales y el resumen final con los mejores resultados obtenidos por cada sector también fueron evaluados y están disponibles como respaldo para futuras comparaciones.

---

### 7.1 Modelo General – Producción en Unidades (Confeccionista y Electrónica)

En esta sección se desarrolla un modelo general para predecir la producción mensual en **unidades** combinando los sectores *Confeccionista* y *Electrónica*. Se emplean técnicas de regresión supervisada y se evalúan tres fases:


#### Variables utilizadas

- **empleados**: cantidad de personal ocupado  
- **establecimientos**: cantidad de establecimientos activos  
- **anio**: componente temporal para capturar tendencias  
- **sector_Electronica**: dummy para distinguir entre los sectores (Confeccionista es la base)

---

In [ ]:
# Copia del dataset general de unidades
df_modelo = df_unidades_conf_elec.copy()

# Creación de variable dummy
df_modelo = pd.get_dummies(df_modelo, columns=["sector"], drop_first=True)

# Variables predictoras y objetivo
X = df_modelo[["empleados", "establecimientos", "anio", "sector_Electronica"]]
y = df_modelo["Produccion"]

# División en entrenamiento y prueba
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarización
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Modelos
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5),
    "Regresión Lineal": LinearRegression()
}

resultados = []
for nombre, modelo in models.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    resultados.append({
        "Modelo": nombre,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R²": round(r2, 3)
    })
# Mostrar resultados
df_resultados_unidades_general = pd.DataFrame(resultados)
display(df_resultados_unidades_general.sort_values(by="R²", ascending=False))


Fase 1 – Modelos lineales regularizados

Se entrenaron modelos de regresión lineal múltiple y variantes con regularización:

- **Regresión Lineal**  
- **Ridge Regression**  
- **Lasso Regression**  
- **ElasticNet**

Los datos fueron estandarizados antes de entrenar. Los resultados mostraron que:

- **Lasso** y **Regresión Lineal** obtuvieron los mejores valores de MAE y R² (0.920)
- **Ridge** también mostró un excelente rendimiento con R² = 0.918
- **ElasticNet** tuvo un rendimiento inferior (R² = 0.809)

Esto indica que los modelos lineales se ajustan muy bien al conjunto combinado, especialmente con una única variable categórica.

---

In [ ]:
# Dataset con dummy ya preparado
df_modelo = df_unidades_conf_elec.copy()
df_modelo = pd.get_dummies(df_modelo, columns=["sector"], drop_first=True)

# Variables predictoras y objetivo
X = df_modelo[["empleados", "establecimientos", "anio", "sector_Electronica"]]
y = df_modelo["Produccion"]

# División en entrenamiento y prueba (ya hecha en Fase 1, se mantiene)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelos no lineales
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

modelos = [
    ("Árbol de Decisión", DecisionTreeRegressor(random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("KNN Regressor", KNeighborsRegressor(n_neighbors=5)),
    ("Radius Neighbors", RadiusNeighborsRegressor(radius=50000))
]

resultados = []
for nombre, modelo in modelos:
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados.append({
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

# Mostrar resultados
df_resultados_no_lineales_unidades = pd.DataFrame(resultados)
display(df_resultados_no_lineales_unidades.sort_values(by="R²", ascending=False))


Fase 2 – Modelos no lineales

Se aplicaron modelos no paramétricos para explorar posibles relaciones complejas y no lineales entre las variables:

- **Árbol de Decisión**  
- **Random Forest**  
- **KNN Regressor**  
- **Radius Neighbors**

En general, estos modelos mostraron un rendimiento **ligeramente inferior** al de los modelos lineales.

- **Random Forest** y **KNN** obtuvieron R² cercanos a 0.91, pero no lograron superar el desempeño de **Lasso** o **Ridge**.
- Además, presentaron una mayor variabilidad entre las predicciones y una menor interpretabilidad.

Estos resultados confirmaron que, para este caso, **los modelos lineales capturan suficientemente bien la estructura de los datos**, sin necesidad de aplicar técnicas más complejas.

---


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import pandas as pd

# Preparar dataset con dummy
df_modelo = df_unidades_conf_elec.copy()
df_modelo = pd.get_dummies(df_modelo, columns=["sector"], drop_first=True)

X = df_modelo[["empleados", "establecimientos", "anio", "sector_Electronica"]]
y = df_modelo["Produccion"]

# Modelos lineales con pipeline
modelos = {
    "Regresión Lineal": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso Regression": make_pipeline(StandardScaler(), Lasso(alpha=1.0)),
    "ElasticNet": make_pipeline(StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5))
}

# Validación cruzada
resultados = []
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X, y, cv=5, scoring='r2')
    resultados.append({
        "Modelo": nombre,
        "R² promedio": round(scores.mean(), 3),
        "Desviación estándar": round(scores.std(), 3)
    })

# Mostrar resultados
df_valcv_unidades = pd.DataFrame(resultados)
display(df_valcv_unidades.sort_values(by="R² promedio", ascending=False))


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import pandas as pd

# Dataset preparado con dummy (reutilizamos df_modelo)
X = df_modelo[["empleados", "establecimientos", "anio", "sector_Electronica"]]
y = df_modelo["Produccion"]

# Modelos no lineales
modelos_nl_cv = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

# Validación cruzada
resultados_nl_cv = []
for nombre, modelo in modelos_nl_cv.items():
    scores = cross_val_score(modelo, X, y, cv=5, scoring='r2')
    resultados_nl_cv.append({
        "Modelo": nombre,
        "R² promedio": round(scores.mean(), 3),
        "Desviación estándar": round(scores.std(), 3)
    })

# Mostrar resultados
df_valcv_nolineal_unidades = pd.DataFrame(resultados_nl_cv)
display(df_valcv_nolineal_unidades.sort_values(by="R² promedio", ascending=False))

Fase 3 – Validación cruzada

Se evaluó la estabilidad de los modelos mediante **K-Fold Cross Validation (k=5)**. Se compararon modelos lineales y no lineales utilizando las mismas variables predictoras.

**Ridge Regression** se destacó como el modelo más equilibrado entre precisión y estabilidad.

---

Conclusión

La combinación de sectores en unidades no afectó negativamente el rendimiento del modelo. Los modelos lineales, en particular **Ridge Regression**, demostraron ser robustos y precisos, con excelente comportamiento en validación cruzada. Por simplicidad y rendimiento, este será el modelo seleccionado en el análisis final.

### 7.2 – Modelo general de predicción para producción en kilogramos

En esta sección se entrena un modelo de regresión supervisada para predecir la producción mensual en kilogramos combinando los sectores **Textil** y **Pesquera**.

Se evaluaron tres fases de modelado para seleccionar el modelo final:
- **Fase 1:** modelos lineales clásicos y regularizados (Ridge, Lasso, ElasticNet)
- **Fase 2:** modelos no lineales (árboles y vecinos)
- **Fase 3:** validación cruzada (5-fold) para comparar robustez y estabilidad

Las variables utilizadas como predictoras fueron:
- `empleados`: cantidad de personal ocupado
- `establecimientos`: cantidad de establecimientos industriales activos
- `anio`: componente temporal para capturar tendencias
- `sector_Textil`: dummy para distinguir entre los sectores (Pesquera es la base)

---

Se probaron varios enfoques y modelos alternativos, y si bien algunos mostraron buen rendimiento inicial, no todos lograron generalizar de forma estable.  
Tras comparar los resultados de todas las fases, se seleccionó **ElasticNet** como modelo final para el conjunto en kilogramos, por ofrecer el mejor equilibrio entre precisión y estabilidad.

A continuación, se muestra el entrenamiento y evaluación de este modelo final.


In [ ]:
# Copia del dataset general de kilogramos
df_modelo_kg = df_kilos_text_pesq.copy()

# Crear variable dummy
df_modelo_kg = pd.get_dummies(df_modelo_kg, columns=["sector"], drop_first=True)

# Variables predictoras y objetivo
X_kg = df_modelo_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]
y_kg = df_modelo_kg["Produccion"]

# División en entrenamiento y prueba
from sklearn.model_selection import train_test_split
X_train_kg, X_test_kg, y_train_kg, y_test_kg = train_test_split(X_kg, y_kg, test_size=0.2, random_state=42)

# Estandarización
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled_kg = scaler.fit_transform(X_train_kg)
X_test_scaled_kg = scaler.transform(X_test_kg)

# Modelos lineales
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5),
    "Regresión Lineal": LinearRegression()
}

resultados = []
for nombre, modelo in models.items():
    modelo.fit(X_train_scaled_kg, y_train_kg)
    y_pred = modelo.predict(X_test_scaled_kg)
    mae = mean_absolute_error(y_test_kg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_kg, y_pred))
    r2 = r2_score(y_test_kg, y_pred)
    resultados.append({
        "Modelo": nombre,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R²": round(r2, 3)
    })

# Mostrar resultados
df_resultados_kg = pd.DataFrame(resultados)
display(df_resultados_kg.sort_values(by="R²", ascending=False))

Fase 1 – Modelos lineales regularizados

Se entrenaron modelos de regresión lineal múltiple y variantes con regularización:

- **Regresión Lineal**
- **Ridge Regression**
- **Lasso Regression**
- **ElasticNet**

Los datos fueron estandarizados antes de entrenar. Los resultados mostraron que:

- **ElasticNet** fue el modelo con mejor desempeño, logrando un **R² = 0.688** y el menor MAE y RMSE entre todos los modelos evaluados.
- **Ridge Regression** también mostró un rendimiento sólido (**R² = 0.683**), apenas por debajo de ElasticNet.
- **Lasso** y **Regresión Lineal** quedaron más relegados con valores cercanos pero inferiores.

Esto indica que, en este conjunto combinado, los modelos lineales capturan bien la relación entre predictores y producción.

---

In [ ]:
# Dataset con dummy preparado
df_modelo_kg = df_kilos_text_pesq.copy()
df_modelo_kg = pd.get_dummies(df_modelo_kg, columns=["sector"], drop_first=True)

# Variables predictoras y objetivo
X_kg = df_modelo_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]
y_kg = df_modelo_kg["Produccion"]

# División en entrenamiento y prueba
X_train_kg, X_test_kg, y_train_kg, y_test_kg = train_test_split(X_kg, y_kg, test_size=0.2, random_state=42)

# Modelos no lineales
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

modelos = [
    ("Árbol de Decisión", DecisionTreeRegressor(random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("KNN Regressor", KNeighborsRegressor(n_neighbors=5)),
    ("Radius Neighbors", RadiusNeighborsRegressor(radius=50000))
]

resultados = []
for nombre, modelo in modelos:
    modelo.fit(X_train_kg, y_train_kg)
    y_pred = modelo.predict(X_test_kg)

    mae = mean_absolute_error(y_test_kg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_kg, y_pred))
    r2 = r2_score(y_test_kg, y_pred)

    resultados.append({
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

# Mostrar resultados
df_resultados_no_lineales_kg = pd.DataFrame(resultados)
display(df_resultados_no_lineales_kg.sort_values(by="R²", ascending=False))


Fase 2 – Modelos no lineales

Se aplicaron modelos no paramétricos para explorar relaciones complejas:

- **Árbol de Decisión**
- **Random Forest**
- **KNN Regressor**
- **Radius Neighbors**

Los modelos **KNN Regressor** y **Random Forest** lograron buenos resultados:

- **KNN** alcanzó un **R² = 0.726**, superando incluso a los modelos lineales.
- **Random Forest** también mostró un **R² = 0.692**, confirmando su capacidad predictiva.

Estos resultados sugieren que existen patrones no lineales relevantes en los datos que pueden ser capturados con enfoques no paramétricos.

---

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
import pandas as pd

# Dataset ya preparado con dummy
X_kg = df_modelo_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]
y_kg = df_modelo_kg["Produccion"]

# Modelos con pipeline
modelos = {
    "Regresión Lineal": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso Regression": make_pipeline(StandardScaler(), Lasso(alpha=1.0)),
    "ElasticNet": make_pipeline(StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5))
}

# Validación cruzada
resultados = []
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_kg, y_kg, cv=5, scoring='r2')
    resultados.append({
        "Modelo": nombre,
        "R² promedio": round(scores.mean(), 3),
        "Desviación estándar": round(scores.std(), 3)
    })

# Mostrar resultados
df_valcv_kg = pd.DataFrame(resultados)
display(df_valcv_kg.sort_values(by="R² promedio", ascending=False))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import pandas as pd

# Dataset ya preparado con dummy
X_kg = df_modelo_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]
y_kg = df_modelo_kg["Produccion"]

# Modelos a validar
modelos_nl_cv = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

# Validación cruzada
resultados_nl_cv = []
for nombre, modelo in modelos_nl_cv.items():
    scores = cross_val_score(modelo, X_kg, y_kg, cv=5, scoring='r2')
    resultados_nl_cv.append({
        "Modelo": nombre,
        "R² promedio": round(scores.mean(), 3),
        "Desviación estándar": round(scores.std(), 3)
    })

# Mostrar resultados
df_valcv_nolineal_kg = pd.DataFrame(resultados_nl_cv)
display(df_valcv_nolineal_kg.sort_values(by="R² promedio", ascending=False))


 Fase 3 – Validación cruzada

Se evaluó la **estabilidad y capacidad de generalización** mediante **K-Fold Cross Validation (k=5)** sobre los modelos lineales y no lineales

Los resultados mostraron que:

- **ElasticNet** fue el  modelo con **R² promedio mas alto (0.317)** y una desviación estándar baja.
- **Random Forest** y **KNN Regressor** obtuvieron **R² altos** pero menores que ElasticNet y mayor desviacion estandar.
- **Ridge**, **Lasso** y **Regresión Lineal** obtuvieron **R² negativos** y mayor variabilidad, indicando sobreajuste y baja generalización.

Conclusión

En el conjunto combinado de kilogramos, **ElasticNet** se posiciona como el modelo más confiable y estable, siendo el único que logró mantener resultados consistentes a lo largo de todas las fases.

Este modelo será el seleccionado para continuar el análisis final sobre los sectores **Textil** y **Pesquera**.

---

## Comparación Final de Modelos

Se resumen aquí los modelos seleccionados para cada sector y conjunto combinado. Los modelos generales fueron seleccionados para el análisis final por su mejor rendimiento en precisión y estabilidad, evaluados tanto por test como por validación cruzada.

| Sector / Conjunto      | Modelo Final     | R² (mejor test) | R² promedio (CV) | Comentario                                                   |
| ---------------------- | ---------------- | --------------- | ---------------- | ------------------------------------------------------------ |
| **Pesquera**           | ElasticNet       | 0.204           | -0.190           | Único modelo con R² positivo; resto con mal desempeño        |
| **Textil**             | Ridge Regression | 0.673           | -1.475           | Mejor R² en test; validación cruzada no favorable en general |
| **Electrónica**        | Ridge Regression | 0.642           | 0.547            | Mejor modelo y más estable en CV                             |
| **Confeccionista**     | ElasticNet       | 0.076           | 0.034            | Mejor entre todos los pobres resultados                      |
| **General Unidades**   | Ridge Regression | 0.918           | 0.872            | Mejor balance entre precisión y estabilidad                  |
| **General Kilogramos** | ElasticNet       | 0.688           | 0.317            | Único modelo robusto en CV                                   |

## 8. Evaluación de los modelos finales

En esta sección se muestran los resultados finales de los modelos generales seleccionados para predecir la producción mensual por sector industrial.

Se evaluaron los dos modelos más robustos entrenados durante la etapa de regresión supervisada:

- **Modelo Ridge**: aplicado al conjunto general de sectores que reportan producción en **unidades** (Electrónica y Confeccionista).
- **Modelo ElasticNet**: aplicado al conjunto general de sectores que reportan producción en **kilogramos** (Textil y Pesquera).

Con el objetivo de mantener una presentación clara y concisa, se muestran en detalle los resultados obtenidos para un sector representativo de cada conjunto:

- **Electrónica**, como ejemplo del modelo Ridge (unidades).
- **Pesquera**, como ejemplo del modelo ElasticNet (kilogramos).

Para cada uno, se presentan los gráficos de evaluación sobre el conjunto de test, incluyendo:

- Comparación de producción real vs. predicha.
- Histograma de errores residuales.
- Gráfico de residuos vs. valores predichos.
- Evolución de producción real y predicha.

Estos resultados permiten verificar la capacidad predictiva de cada modelo y su comportamiento sobre sectores reales.


### Modelo para sectores en unidades

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Definimos de nuevo los datos X e y y su división, si no estan activos
X = df_modelo[["empleados", "establecimientos", "anio", "sector_Electronica"]]
y = df_modelo["Produccion"]

X_train_uni, X_test_uni, y_train_uni, y_test_uni = train_test_split(X, y, test_size=0.2, random_state=42)

# Reentrenar Ridge con escalado
modelo_ridge_general = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
modelo_ridge_general.fit(X_train_uni, y_train_uni)

# Predicción
y_pred_uni = modelo_ridge_general.predict(X_test_uni)

X_test_con_sector = df_unidades_conf_elec.iloc[X_test_uni.index].copy()
X_test_con_sector = X_test_con_sector.reset_index(drop=True)

# Crear un DataFrame con resultados
df_resultado_sector = pd.DataFrame({
    "Produccion_real": y_test_uni.values,
    "Produccion_predicha": y_pred_uni,
    "sector": X_test_con_sector["sector"]
})

# Filtrar solo Electrónica
df_electronica = df_resultado_sector[df_resultado_sector["sector"] == "Electronica"].reset_index(drop=True)

#### Real vs Predicho – Gráfico de dispersión

In [ ]:
sns.scatterplot(x="Produccion_real", y="Produccion_predicha", data=df_electronica)
plt.plot(
    [df_electronica["Produccion_real"].min(), df_electronica["Produccion_real"].max()],
    [df_electronica["Produccion_real"].min(), df_electronica["Produccion_real"].max()],
    color='red', linestyle='--', label='Perfecto ajuste'
)
plt.xlabel("Producción real (Electrónica)")
plt.ylabel("Producción predicha")
plt.title("Real vs Predicho – Electrónica (modelo Ridge)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico compara directamente los valores reales de producción con las predicciones del modelo.

- Idealmente, los puntos deberían alinearse sobre la diagonal roja (línea de ajuste perfecto).
- Se observa que la mayoría de los puntos siguen un patrón cercano a esa línea, lo que indica un buen desempeño del modelo.
- Algunas desviaciones aparecen en valores altos, pero no afectan significativamente el ajuste general.

#### Histograma de errores residuales

In [ ]:
# Error sector Electrónica
df_electronica["residuo"] = df_electronica["Produccion_real"] - df_electronica["Produccion_predicha"]

sns.histplot(df_electronica["residuo"], kde=True, bins=20, color="orange")
plt.axvline(0, color="black", linestyle="--")
plt.title("Histograma de errores residuales – Electrónica")
plt.xlabel("Error (real - predicho)")
plt.grid(True)
plt.tight_layout()
plt.show()

Aquí se analiza la distribución de los errores (producción real − predicha):

- La forma del histograma es razonablemente simétrica, con una leve desviación a la derecha.
- La mayor parte de los errores se concentran cerca de cero, lo cual es deseable.
- La curva KDE permite visualizar la densidad y tendencia general de los residuos.

#### Gráfico de residuos vs valores predichos

In [ ]:
sns.scatterplot(x="Produccion_predicha", y="residuo", data=df_electronica, color="orange")
plt.axhline(0, color="black", linestyle="--")
plt.title("Residuos vs Producción Predicha – Electrónica")
plt.xlabel("Producción predicha")
plt.ylabel("Error (real - predicho)")
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico es útil para detectar patrones no capturados por el modelo (heterocedasticidad, no linealidades):

- Los residuos están distribuidos alrededor de la línea horizontal en cero.
- No se observan patrones sistemáticos, lo que sugiere que el modelo no deja relaciones estructurales sin capturar.
- Confirma que los errores no dependen directamente del valor predicho.

#### Predicción vs Real – Sector Electrónica (modelo Ridge General)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_electronica["Produccion_real"], label="Producción real", marker="o")
plt.plot(df_electronica["Produccion_predicha"], label="Predicción del modelo", linestyle='--', marker="x")
plt.title("Predicción vs Real – Sector Electrónica (Ridge)")
plt.ylabel("Producción (unidades)")
plt.xlabel("Observaciones del sector (test)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

El gráfico muestra que el modelo Ridge entrenado sobre el conjunto general de unidades predice con notable precisión la producción del sector Electrónica.

Se observa que:

- Las curvas de producción real y predicha siguen un patrón muy similar, especialmente en los picos altos y valles bajos.
- Las discrepancias son bajas y consistentes, lo que indica que el modelo logra capturar correctamente la dinámica de variación mensual en este sector.
- La variable categórica sector_Electronica, incorporada como dummy en el entrenamiento, fue suficiente para permitirle al modelo ajustar bien este subconjunto específico.

Este resultado confirma que el modelo general no solo funciona bien en promedio, sino que es altamente efectivo para el sector Electrónica en particular.

### Modelo sectores en KG

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df_modelo_kg = df_modelo_kg.reset_index(drop=True)


# Redefinir X e y para kilogramos
X_kg = df_modelo_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]
y_kg = df_modelo_kg["Produccion"]


# División en entrenamiento y prueba
X_train_kg, X_test_kg, y_train_kg, y_test_kg = train_test_split(X_kg, y_kg, test_size=0.2, random_state=42)

# Reentrenar ElasticNet con estandarización
modelo_elasticnet_kg = make_pipeline(StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5))
modelo_elasticnet_kg.fit(X_train_kg, y_train_kg)

# Predicción
y_pred_kg = modelo_elasticnet_kg.predict(X_test_kg)

# Obtener sectores correspondientes al conjunto de test
X_test_con_sector_kg = df_kilos_text_pesq.iloc[X_test_kg.index].copy()
X_test_con_sector_kg = X_test_con_sector_kg.reset_index(drop=True)

# Crear DataFrame con predicciones y sectores
df_resultado_kg = pd.DataFrame({
    "Produccion_real": y_test_kg.values,
    "Produccion_predicha": y_pred_kg,
    "sector": X_test_con_sector_kg["sector"]
})

# Filtrar por sector
df_textil = df_resultado_kg[df_resultado_kg["sector"] == "Textil"].reset_index(drop=True)
df_pesquera = df_resultado_kg[df_resultado_kg["sector"] == "Pesquera"].reset_index(drop=True)

#### Real vs Predicho – Gráfico de dispersión

In [ ]:
sns.scatterplot(x="Produccion_real", y="Produccion_predicha", data=df_pesquera, color="darkgreen")
plt.plot(
    [df_pesquera["Produccion_real"].min(), df_pesquera["Produccion_real"].max()],
    [df_pesquera["Produccion_real"].min(), df_pesquera["Produccion_real"].max()],
    color='red', linestyle='--', label='Perfecto ajuste'
)
plt.xlabel("Producción real (Pesquera)")
plt.ylabel("Producción predicha")
plt.title("Real vs Predicho – Pesquera (modelo ElasticNet)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico compara la producción mensual real del sector pesquero con los valores estimados por el modelo ElasticNet.

- La línea roja representa un ajuste perfecto, donde los valores reales y predichos coinciden.
- En general, los puntos siguen una tendencia cercana a la línea ideal, aunque con mayor dispersión en valores bajos.
- Se observa que el modelo logra capturar razonablemente el comportamiento de la producción, especialmente en los niveles intermedios.
- Las desviaciones son visibles, pero se mantienen dentro de un rango aceptable, lo cual valida el uso de ElasticNet en este sector con datos más limitados.

Este resultado refuerza que el modelo general en kilogramos puede adaptarse a sectores con menor cantidad de registros, sin perder capacidad de predicción.

#### Histograma de errores residuales

In [ ]:
# Calcular el residuo si no está creado
df_pesquera["residuo"] = df_pesquera["Produccion_real"] - df_pesquera["Produccion_predicha"]

sns.histplot(df_pesquera["residuo"], kde=True, bins=20, color="darkgreen")
plt.axvline(0, color="black", linestyle="--")
plt.title("Histograma de errores residuales – Pesquera")
plt.xlabel("Error (real - predicho)")
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico muestra la distribución de los errores residuales (diferencia entre la producción real y la predicha) para el sector Pesquera, utilizando el modelo ElasticNet.

- La mayoría de los errores se agrupan cerca del cero, lo que indica que las predicciones no se desvían mucho del valor real.
- La curva KDE sugiere una distribución simétrica y razonablemente centrada, aunque con colas algo más extendidas.
- No se observan errores extremos o sesgos evidentes hacia sobreestimación o subestimación.

Este análisis confirma que el modelo logra un nivel aceptable de ajuste en este sector, incluso considerando su menor volumen de datos.

#### Gráfico de residuos vs valores predichos

In [ ]:
sns.scatterplot(x="Produccion_predicha", y="residuo", data=df_pesquera, color="darkgreen")
plt.axhline(0, color="black", linestyle="--")
plt.title("Residuos vs Producción Predicha – Pesquera")
plt.xlabel("Producción predicha")
plt.ylabel("Error (real - predicho)")
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico permite visualizar si los errores de predicción presentan algún patrón sistemático en función de los valores predichos por el modelo ElasticNet.

- La dispersión de los residuos se encuentra centrada alrededor de cero, lo cual es deseable en un modelo de regresión.
- No se evidencian patrones curvos o estructuras claras, lo que sugiere que el modelo no dejó relaciones no lineales importantes sin capturar.
- Aunque hay mayor variabilidad en algunos rangos de producción, los errores se mantienen dentro de un rango aceptable.

Este comportamiento indica que el modelo general logra un ajuste razonable para el sector Pesquera, sin sesgos evidentes ni problemas graves de heterocedasticidad.

#### Predicción vs Real – Sector Confeccionista (modelo Ridge General)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df_pesquera["Produccion_real"], label="Producción real", marker="o")
plt.plot(df_pesquera["Produccion_predicha"], label="Predicción del modelo", linestyle='--', marker="x")
plt.title("Predicción vs Real – Sector Pesquera (modelo ElasticNet)")
plt.ylabel("Producción (kilogramos)")
plt.xlabel("Observaciones del sector (test)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Este gráfico permite comparar visualmente los valores reales de producción mensual con las predicciones generadas por el modelo ElasticNet aplicado al sector Pesquera.

- Las curvas de producción real y predicha siguen una trayectoria similar, con coincidencias claras en varias crestas y valles del comportamiento mensual.
- Se observan diferencias moderadas en algunos puntos, pero sin errores extremos o predicciones inestables.
- A pesar de las fluctuaciones del sector, el modelo logra mantener una predicción razonablemente ajustada al patrón histórico.

Este resultado respalda que, dentro de las limitaciones del conjunto de datos, el modelo general en kilogramos puede capturar adecuadamente la dinámica mensual del sector Pesquera.

## 9 - Aplicación del modelo final sobre datos reales sin producción

En esta sección se aplican los modelos generales entrenados (Ridge para unidades y ElasticNet para kilogramos) sobre un conjunto externo de datos históricos reales (`df_test`), que no contienen valores en la variable objetivo `Producción`.

Este conjunto abarca el período 2001–2013 y posee únicamente variables explicativas (`empleados`, `establecimientos`, `año` y `sector`). El objetivo de esta etapa es **simular la producción estimada en períodos sin datos reales**, validando así la capacidad de los modelos para generar predicciones coherentes a partir de estructuras industriales básicas.


Se presentan las predicciones obtenidas para los sectores representativos de cada conjunto, junto con visualizaciones que permiten analizar su comportamiento y consistencia temporal.


### Modelo Ridge para sectores con produccion en unidades

In [ ]:
# Filtrar solo sectores incluidos en el modelo final
df_test_filtrado = df_test[df_test["sector"].isin(["Confeccionista", "Electronica"])].copy()

# Crear dummy para el sector (sector_Electronica)
df_test_filtrado = pd.get_dummies(df_test_filtrado, columns=["sector"], drop_first=True)

# Seleccionar variables predictoras
X_test_pred = df_test_filtrado[["empleados", "establecimientos", "anio", "sector_Electronica"]]

# Aplicar predicción con el modelo final entrenado
df_test_filtrado["Produccion_predicha"] = modelo_ridge_general.predict(X_test_pred)

# Redondear producción predicha a entero
df_test_filtrado["Produccion_predicha"] = df_test_filtrado["Produccion_predicha"].round().astype(int)

# Vista previa de resultados
df_test_filtrado[["anio", "mes", "empleados", "establecimientos", "sector_Electronica", "Produccion_predicha"]].head()

In [ ]:
# Crear columna de fecha para graficar
meses_map = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04', 'mayo': '05', 'junio': '06',
    'julio': '07', 'agosto': '08', 'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

df_test_filtrado["mes_num"] = df_test_filtrado["mes"].str.lower().map(meses_map)
df_test_filtrado["fecha"] = pd.to_datetime(df_test_filtrado["anio"].astype(str) + "-" + df_test_filtrado["mes_num"] + "-01")
df_test_filtrado = df_test_filtrado.sort_values("fecha")

# Crear una columna más legible para el gráfico
df_test_filtrado["Sector"] = df_test_filtrado["sector_Electronica"].map({
    True: "Electrónica (naranja)",
    False: "Confeccionista (azul)"
})

# Gráfico
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_test_filtrado, x="fecha", y="Produccion_predicha", hue="Sector")
plt.title("Producción mensual predicha por sector (modelo Ridge – Unidades)")
plt.xlabel("Fecha")
plt.ylabel("Producción estimada (unidades)")
plt.grid(True)
plt.tight_layout()
plt.show()

#### Predicción de producción en sectores reales con modelo general de unidades (Ridge)

Se muestra la predicción realizada con el modelo **Ridge Regression** entrenado sobre el conjunto general de producción en unidades (sectores Electrónica y Confeccionista), aplicado ahora sobre un conjunto de datos reales **sin información de producción** (desde el año 2001 hasta 2013).

Este conjunto contiene datos históricos sobre empleo, cantidad de establecimientos y sector, que permiten estimar la producción mensual a partir del modelo previamente entrenado.

Se observa la evolución estimada de la producción en ambos sectores:

##### Gráfico de predicción mensual por sector

- **Línea azul**: Sector *Confeccionista*  
- **Línea naranja**: Sector *Electrónica*

Este gráfico permite observar la dinámica mensual estimada en cada sector a lo largo del tiempo.

---

##### Observaciones:

- El modelo predice valores consistentes para el sector *Confeccionista*, con una línea estable en el tiempo, acorde a lo observado en los datos de entrenamiento.
- Para el sector *Electrónica*, se evidencia una evolución creciente en la producción estimada, lo cual coincide con un aumento progresivo de empleados y establecimientos en dicho sector en las décadas recientes.
- Se logra entonces una predicción diferenciada y estructuralmente coherente entre ambos sectores, usando como única entrada los datos históricos de personal ocupado, establecimientos y tiempo.

---

Este análisis valida el uso del modelo Ridge General como una herramienta válida para realizar simulaciones o proyecciones sobre datos reales donde no se dispone de la variable objetivo (`Producción`).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Gráfico 1: Producción predicha vs empleados
plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=df_test_filtrado,
    x="empleados", y="Produccion_predicha",
    hue="sector_Electronica",
    palette={False: "steelblue", True: "orange"},
    legend=False  # Ocultamos leyenda automática
)
plt.title("Producción predicha vs Empleados")
plt.xlabel("Cantidad de empleados")
plt.ylabel("Producción estimada (unidades)")
# Leyenda manual con colores correctos
plt.scatter([], [], color="steelblue", label="Confeccionista (azul)")
plt.scatter([], [], color="orange", label="Electrónica (naranja)")
plt.legend(title="Sector")
plt.grid(True)
plt.tight_layout()
plt.show()

# Gráfico 2: Producción predicha vs establecimientos
plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=df_test_filtrado,
    x="establecimientos", y="Produccion_predicha",
    hue="sector_Electronica",
    palette={False: "steelblue", True: "orange"},
    legend=False
)
plt.title("Producción predicha vs Establecimientos")
plt.xlabel("Cantidad de establecimientos")
plt.ylabel("Producción estimada (unidades)")
plt.scatter([], [], color="steelblue", label="Confeccionista (azul)")
plt.scatter([], [], color="orange", label="Electrónica (naranja)")
plt.legend(title="Sector")
plt.grid(True)
plt.tight_layout()
plt.show()


#### Producción predicha vs Empleados (modelo Ridge – Unidades)

Este gráfico muestra la relación entre la **cantidad de empleados** y la **producción mensual predicha** por el modelo general de unidades para los sectores **Confeccionista** (azul) y **Electrónica** (naranja).

Se observa una tendencia clara en ambos sectores:
- En **Electrónica**, la producción estimada crece de forma casi lineal con la cantidad de empleados, indicando una fuerte relación entre ambas variables.
- En **Confeccionista**, los valores están más concentrados y la producción estimada varía poco, lo cual coincide con su estructura más acotada y estable.

---


#### Producción predicha vs Establecimientos (modelo Ridge – Unidades)

En este gráfico se analiza cómo varía la producción mensual estimada en función de la **cantidad de establecimientos industriales** en los sectores **Confeccionista** (azul) y **Electrónica** (naranja).

Los patrones observados son:
- En el sector **Electrónica**, se mantiene una clara correlación positiva: mayor cantidad de establecimientos tiende a asociarse con mayor producción.
- En el sector **Confeccionista**, la producción se mantiene en un rango más estable, con poca variación en los niveles de establecimientos.

Estos resultados reafirman que el modelo capta correctamente las diferencias estructurales entre sectores, utilizando las variables históricas como buenos predictores.

---

### Modelo ElasticNet para sectores con produccion en KG

In [ ]:
# Filtrar solo sectores incluidos en el modelo final
df_test_filtrado_kg = df_test[df_test["sector"].isin(["Textil", "Pesquera"])].copy()

# Crear dummy para el sector (sector_Textil)
df_test_filtrado_kg = pd.get_dummies(df_test_filtrado_kg, columns=["sector"], drop_first=True)

# Seleccionar variables predictoras
X_test_pred_kg = df_test_filtrado_kg[["empleados", "establecimientos", "anio", "sector_Textil"]]

# Aplicar predicción con el modelo final entrenado
df_test_filtrado_kg["Produccion_predicha"] = modelo_elasticnet_kg.predict(X_test_pred_kg)

# Redondear producción predicha a enteros
df_test_filtrado_kg["Produccion_predicha"] = df_test_filtrado_kg["Produccion_predicha"].round().astype(int)

# Vista previa de resultados
df_test_filtrado_kg[["anio", "mes", "empleados", "establecimientos", "sector_Textil", "Produccion_predicha"]].head()


In [ ]:
# Crear columna de fecha para graficar
meses_map = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04', 'mayo': '05', 'junio': '06',
    'julio': '07', 'agosto': '08', 'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

df_test_filtrado_kg["mes_num"] = df_test_filtrado_kg["mes"].str.lower().map(meses_map)
df_test_filtrado_kg["fecha"] = pd.to_datetime(df_test_filtrado_kg["anio"].astype(str) + "-" + df_test_filtrado_kg["mes_num"] + "-01")
df_test_filtrado_kg = df_test_filtrado_kg.sort_values("fecha")

# Crear una columna más legible para el gráfico
df_test_filtrado_kg["Sector"] = df_test_filtrado_kg["sector_Textil"].map({
    True: "Textil (naranja)",
    False: "Pesquera (Azul)"
})

In [ ]:
# Gráfico de producción mensual predicha por sector
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_test_filtrado_kg, x="fecha", y="Produccion_predicha", hue="Sector")
plt.title("Producción mensual predicha por sector (modelo ElasticNet – Kilogramos)")
plt.xlabel("Fecha")
plt.ylabel("Producción estimada (kilogramos)")
plt.grid(True)
plt.tight_layout()
plt.show()


#### Predicción de producción en sectores reales con modelo general de kilogramos (ElasticNet)

Se muestra la predicción realizada con el modelo **ElasticNet** entrenado sobre el conjunto general de producción en kilogramos (sectores **Textil** y **Pesquera**), aplicado ahora sobre un conjunto de datos reales sin información de producción (desde el año 2001 hasta 2013).

Este conjunto contiene datos históricos sobre empleo, cantidad de establecimientos y sector, que permiten estimar la producción mensual a partir del modelo previamente entrenado.

Se observa la evolución estimada de la producción en ambos sectores:

##### Gráfico de predicción mensual por sector

- **Línea azul**: Sector *Pesquera*  
- **Línea naranja**: Sector *Textil*

Este gráfico permite observar la dinámica mensual estimada en cada sector a lo largo del tiempo.

---

#### Observaciones:

- El modelo predice valores razonables para el sector **Pesquera**, con una curva que muestra variabilidad y cierta estacionalidad, en línea con los registros del entrenamiento.
- En el sector **Textil**, se evidencia una producción más alta y más estable en el tiempo, coincidiendo con su mayor cantidad de empleados y establecimientos históricos.
- Se logra entonces una **predicción diferenciada coherente** entre ambos sectores, utilizando únicamente datos estructurales como insumo.

Este análisis valida el uso del modelo **ElasticNet** como una herramienta útil para realizar simulaciones o proyecciones sobre datos reales donde no se dispone de la variable objetivo (`Producción`).


In [ ]:
# Gráfico 1: Producción predicha vs empleados
plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=df_test_filtrado_kg,
    x="empleados", y="Produccion_predicha",
    hue="sector_Textil",
    palette={False: "darkgreen", True: "mediumblue"},
    legend=False  # Ocultamos leyenda automática
)

# Leyenda manual con colores correctos
plt.scatter([], [], color="darkgreen", label="Pesquera (verde)")
plt.scatter([], [], color="mediumblue", label="Textil (azul)")

plt.legend(title="Sector")
plt.title("Producción predicha vs Empleados")
plt.xlabel("Cantidad de empleados")
plt.ylabel("Producción estimada (kilogramos)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2: Producción predicha vs establecimientos
plt.figure(figsize=(12, 5))
sns.scatterplot(
    data=df_test_filtrado_kg,
    x="establecimientos", y="Produccion_predicha",
    hue="sector_Textil",
    palette={False: "darkgreen", True: "mediumblue"},
    legend=False
)

plt.scatter([], [], color="darkgreen", label="Pesquera (verde)")
plt.scatter([], [], color="mediumblue", label="Textil (azul)")

plt.legend(title="Sector")
plt.title("Producción predicha vs Establecimientos")
plt.xlabel("Cantidad de establecimientos")
plt.ylabel("Producción estimada (kilogramos)")
plt.grid(True)
plt.tight_layout()
plt.show()


#### Producción predicha vs Empleados (modelo ElasticNet – Kilogramos)

Este gráfico muestra la relación entre la **cantidad de empleados** y la **producción mensual predicha** por el modelo general en kilogramos para los sectores **Pesquera** (verde) y **Textil** (azul).

Se observan los siguientes patrones:

- En el sector **Textil**, la producción predicha se mantiene en un rango elevado y con escasa dispersión, lo que indica un comportamiento más estructurado y regular frente a los cambios en el empleo.
- En **Pesquera**, la producción predicha muestra una mayor dispersión a lo largo del eje de empleados, reflejando la variabilidad interna de este sector en los niveles de personal, aunque con una pendiente creciente coherente.

---

#### Producción predicha vs Establecimientos (modelo ElasticNet – Kilogramos)

Este gráfico permite analizar cómo varía la producción mensual estimada en función de la **cantidad de establecimientos industriales** para los sectores **Pesquera** (verde) y **Textil** (azul).

Los resultados muestran:

- En **Textil**, los valores de producción se concentran alrededor de 7 y 8 establecimientos, con una producción estimada bastante homogénea y elevada, lo cual coincide con su estructura estable.
- En **Pesquera**, la producción muestra una ligera tendencia creciente a medida que aumentan los establecimientos, pero los puntos son más dispersos y la variabilidad es mayor.

Estos patrones refuerzan que el modelo mantiene las relaciones estructurales observadas en los datos reales y que logra predecir con solidez a partir de variables históricas básicas del sector.

### Esta simulación final valida que los modelos seleccionados pueden extrapolar con coherencia sobre datos históricos sin producción, utilizando únicamente estructuras industriales básicas. Se observan patrones diferenciados entre sectores, lo que demuestra la capacidad del modelo para captar la dinámica interna incluso en escenarios sin etiquetas. Esto respalda su uso futuro en tareas de predicción o análisis histórico.